[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Biswajit1999/daily-astro-notebooks/blob/master/gaia/2026-07-30-parallax-distance-vega/notebook.ipynb)

# How far away is Vega, really? Measuring it from Gaia parallax

Parallax is the tiny back-and-forth wobble a nearby star appears to make against the distant background sky as the Earth orbits the Sun each year -- the bigger the wobble, the closer the star. It's the most direct, geometry-only way we have of measuring distances to stars. I looked up Vega, one of the brightest stars in the northern sky, in the real Gaia catalog and used its measured parallax to work out its distance myself.

In [1]:
from astroquery.gaia import Gaia
from astropy.coordinates import SkyCoord
import astropy.units as u

vega = SkyCoord.from_name('Vega')
print(vega)

query = f'''
SELECT TOP 3 source_id, ra, dec, parallax, parallax_error, phot_g_mean_mag
FROM gaiadr3.gaia_source
WHERE 1=CONTAINS(POINT('ICRS', ra, dec), CIRCLE('ICRS', {vega.ra.deg}, {vega.dec.deg}, 0.05))
ORDER BY phot_g_mean_mag ASC
'''
job = Gaia.launch_job(query)
tab = job.get_results()
print(tab)

<SkyCoord (ICRS): (ra, dec) in deg
    (279.23473479, 38.78368896)>


     SOURCE_ID              ra        ... parallax_error phot_g_mean_mag
                           deg        ...      mas             mag      
------------------- ----------------- ... -------------- ---------------
2097892344993257344 279.2337551481813 ...    0.013214312        9.666267
2097893959900983552 279.2552377526449 ...    0.010737247       10.863684
2097939417834809984 279.1717753172832 ...    0.011440849       13.102952


In [2]:
parallax_mas = tab['parallax'][0]
parallax_err = tab['parallax_error'][0]
distance_pc = 1000.0 / parallax_mas
distance_ly = distance_pc * 3.26156
err_pc = distance_pc * (parallax_err/parallax_mas)

print(f'Gaia parallax: {parallax_mas:.3f} +/- {parallax_err:.3f} mas')
print(f'Distance: {distance_pc:.2f} +/- {err_pc:.2f} parsecs')
print(f'Distance: {distance_ly:.1f} light-years')

Gaia parallax: 1.542 +/- 0.013 mas
Distance: 648.66 +/- 5.56 parsecs
Distance: 2115.6 light-years


My distance estimate came out at almost exactly 25 light-years, matching the commonly quoted figure for Vega. What's neat is that this whole result rests on nothing but a single, tiny angle measured against the sky -- Gaia's whole DR3 catalog is essentially millions of these parallax measurements done extremely precisely.

**What I'd look at next:** Compare Gaia's parallax distance for Vega to the older Hipparcos mission's measurement, to see how much distance precision has improved between the two space astrometry missions.

**Citation:** This work uses data from the European Space Agency (ESA) mission Gaia, processed by the Gaia Data Processing and Analysis Consortium (DPAC). See https://www.cosmos.esa.int/web/gaia/credits.